### Importation des modules

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder

### Chargement des données

In [ ]:
# Optionnel mais pratique pour l'affichage dans Jupyter
pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")

# Charger les informations des chiens
df_dogs = pd.read_excel(r"C:\Users\Cours APT\Documents\dog-behavior-analysis\data\DogInfo.xlsx")
print(f"Nombre de chiens : {len(df_dogs)}")
display(df_dogs.head())

# Charger les séries temporelles relatives à chaque chien / test
na_values = ['<undefined>']

df_ts = pd.read_csv(
    r"C:\Users\Cours APT\Documents\dog-behavior-analysis\data\DogMoveData.csv", 
    na_values=na_values,
    low_memory=False
)

print(f"Dimensions de l'échantillon : {df_ts.shape}")
display(df_ts.head())

### Séparation des jeux en train / test + vérification de l'équilibre des classes

In [ ]:
# 1. On exclut les lignes non labellisées pour se concentrer sur les données utiles
df_annotated = df_ts.dropna(subset=['Behavior_1']).copy()

# 2. Définition du split 80/20 basé sur les groupes (DogID)
# Le random_state garantit que tu obtiendras toujours le même split en relançant la cellule
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=2)

# Récupération des indices pour le Train et le Test
train_idx, test_idx = next(gss.split(df_annotated, groups=df_annotated['DogID']))

df_train = df_annotated.iloc[train_idx]
df_test = df_annotated.iloc[test_idx]

# Vérification du nombre de chiens
chiens_train = df_train['DogID'].nunique()
chiens_test = df_test['DogID'].nunique()
print(f"=== Répartition des {chiens_train + chiens_test} chiens ===")
print(f"Train : {chiens_train} chiens ({(chiens_train/(chiens_train+chiens_test)*100):.1f}%)")
print(f"Test  : {chiens_test} chiens ({(chiens_test/(chiens_train+chiens_test)*100):.1f}%)")
print("-" * 50)

# 3. Calcul et comparaison des proportions de chaque comportement
prop_train = df_train['Behavior_1'].value_counts(normalize=True) * 100
prop_test = df_test['Behavior_1'].value_counts(normalize=True) * 100

# Création d'un DataFrame de comparaison
df_compare = pd.DataFrame({'Train (%)': prop_train, 'Test (%)': prop_test}).fillna(0)
df_compare = df_compare.sort_values(by='Train (%)', ascending=False).round(2)

print("\n=== Proportions des classes dans Train et Test ===")
display(df_compare)

# 4. Visualisation graphique de l'équilibre
df_compare.plot(kind='bar', figsize=(15, 6), color=['#3498db', '#e74c3c'], alpha=0.8)
plt.title("Distribution des comportements (Train vs Test)", fontsize=14)
plt.ylabel("Pourcentage des données annotées (%)")
plt.xlabel("Comportements")
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# 1. Isoler les ID des chiens présents dans chaque ensemble
train_ids = df_train['DogID'].unique()
test_ids = df_test['DogID'].unique()

# 2. Récupérer leurs caractéristiques depuis le dataset statique
df_dogs_train = df_dogs[df_dogs['DogID'].isin(train_ids)].copy()
df_dogs_test = df_dogs[df_dogs['DogID'].isin(test_ids)].copy()

# Ajouter une colonne pour identifier la provenance
df_dogs_train['Set'] = 'Train'
df_dogs_test['Set'] = 'Test'

# Fusionner pour la visualisation
df_dogs_compare = pd.concat([df_dogs_train, df_dogs_test])

# 3. Tableau récapitulatif rapide des moyennes
print("=== Comparaison des profils physiques (Moyennes) ===")
display(df_dogs_compare.groupby('Set')[['Age months', 'Weight']].mean().round(1))

# 4. Visualisation comparative
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Graphique A : Poids
sns.boxplot(data=df_dogs_compare, x='Set', y='Weight', hue='Set', legend=False, ax=axes[0, 0], palette=['#3498db', '#e74c3c'], width=0.4)
sns.stripplot(data=df_dogs_compare, x='Set', y='Weight', ax=axes[0, 0], color='black', alpha=0.5, jitter=True)
axes[0, 0].set_title("Distribution du Poids (Train vs Test)")
axes[0, 0].set_ylabel("Poids")

# Graphique B : Âge 
sns.boxplot(data=df_dogs_compare, x='Set', y='Age months', hue='Set', legend=False, ax=axes[0, 1], palette=['#3498db', '#e74c3c'], width=0.4)
sns.stripplot(data=df_dogs_compare, x='Set', y='Age months', ax=axes[0, 1], color='black', alpha=0.5, jitter=True)
axes[0, 1].set_title("Distribution de l'Âge (Train vs Test)")
axes[0, 1].set_ylabel("Âge (mois)")

# Graphique C : Genre (Correction de la méthode de calcul du pourcentage)
gender_counts = df_dogs_compare.groupby(['Set', 'Gender']).size().reset_index(name='Count')
set_totals = df_dogs_compare.groupby('Set').size().reset_index(name='Total')
gender_prop = pd.merge(gender_counts, set_totals, on='Set')
gender_prop['Pourcentage'] = 100 * gender_prop['Count'] / gender_prop['Total']

sns.barplot(data=gender_prop, x='Set', y='Pourcentage', hue='Gender', ax=axes[1, 0], palette='Set2')
axes[1, 0].set_title("Répartition du Genre (%)")
axes[1, 0].set_ylabel("Pourcentage (%)")

# Graphique D : Toutes les Races 
# On récupère toutes les races existantes, triées par ordre décroissant
all_breeds = df_dogs_compare['Breed'].value_counts().index

sns.countplot(
    data=df_dogs_compare, 
    y='Breed', 
    hue='Set', 
    ax=axes[1, 1], 
    palette=['#3498db', '#e74c3c'], 
    order=all_breeds
)
axes[1, 1].set_title("Toutes les Races présentes (Train vs Test)")
axes[1, 1].set_xlabel("Nombre de chiens")
axes[1, 1].set_ylabel("")

# Réduction de la taille de la police pour accommoder les 26 races
axes[1, 1].tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.show()

# 1. Identification des chiens dans chaque set
train_ids = df_train['DogID'].unique()
test_ids = df_test['DogID'].unique()

# 2. Assignation du set à chaque chien dans df_dogs
df_dogs_tailles = df_dogs.copy()
df_dogs_tailles['Set'] = df_dogs_tailles['DogID'].apply(
    lambda x: 'Train' if x in train_ids else ('Test' if x in test_ids else 'Ignoré')
)
df_dogs_tailles = df_dogs_tailles[df_dogs_tailles['Set'].isin(['Train', 'Test'])]

# 3. Calcul des effectifs par catégorie et par set
colonnes_tailles = [
    'Taille_Categorie_Taille_Petite', 
    'Taille_Categorie_Taille_Moyenne', 
    'Taille_Categorie_Taille_Grande'
]
repartition_tailles = df_dogs_tailles.groupby('Set')[colonnes_tailles].sum().astype(int)

print("=== Nombre de chiens par catégorie de taille (Train vs Test) ===")
display(repartition_tailles)

# 4. Transformation au format long pour le graphique
df_melted = repartition_tailles.reset_index().melt(
    id_vars='Set', 
    var_name='Catégorie', 
    value_name='Nombre de chiens'
)

# Nettoyage des noms pour rendre le graphique plus lisible
df_melted['Catégorie'] = df_melted['Catégorie'].str.replace('Taille_Categorie_', '').str.replace('_', ' ')

# 5. Visualisation
plt.figure(figsize=(10, 6))
sns.barplot(data=df_melted, x='Catégorie', y='Nombre de chiens', hue='Set', palette=['#3498db', '#e74c3c'])
plt.title("Répartition des tailles de chiens par sous-ensemble", fontsize=14)
plt.ylabel("Nombre de chiens uniques")
plt.xlabel("Catégorie de taille")
plt.legend(title='Set')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()